# Genomics Laboratory

## Student Notebook

This notebook contains exercises for the Genomics laboratory session. Complete each exercise by following the step-by-step instructions and filling in the code stubs.

**Learning Objectives:**
- Retrieve genomic data from public databases using REST APIs
- Analyze DNA sequences and compute basic statistics
- Find Transcription-factor binging sites using knonwn binding motifs
- Integrate multi-omics data (genomics, epigenomics)

**Instructions:**
- Read each exercise description carefully
- Follow the step-by-step instructions
- Fill in the code stubs marked with `# TODO:` comments
- Run each cell and verify your results
- Ask for help if you get stuck!

---

## Laboratory Pipeline Overview

1. **Exercise 1: Calling motifs**
   - Find the information about the CTCF binding motif in mouse (_Mus musculus_).
   - Download and inspect the Jaspar files.
   - Plot motif score count and distribution.
   - Select a set of significant motifs. Save their locations to a .bed file along with sequence information.

1. **Exercise 2: Validation using ChIP-seq data**
   - Find the ChIP-seq signal data for the **relevant dataset**.
   - Use the genome browser to preview the signal.
   - Download the significant peaks (we will skep _peak calling_).
   - How well do the identified motifs correspond to the actual ChIP-seq peaks?
   
      - Map peaks to motifs (use `pyranges`).
      - Compute overlaps between peaks and motifs. Display 2x2 table of mapped / unmapped peaks / motifs.
      - Plot motif score vs peak strength
   
   - Prepare and save a "gold" .bed file with significant motifs that also have strong peaks inside.
   

In [ ]:
# Import required libraries
%matplotlib inline
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from scipy.integrate import solve_ivp
import os

# Configure plotting
plt.style.use('seaborn-v0_8')

# Set up output directory
output_dir = "lab_outputs"
os.makedirs(output_dir, exist_ok=True)

print("✓ Libraries imported successfully")


## Exercise 1: Calling Motifs

**Goal:** Teach motif databases usage, PWM/PSSM handling, motif scanning, and genomic interval handling.

**Background:**
Transcription factors bind DNA at specific sequence patterns called *motifs*. These motifs are often represented as Position Weight Matrices (PWMs), which describe nucleotide preferences at each position. Databases like JASPAR provide curated motif collections, including the well-known insulator protein CTCF binding motif in humans.

---

### **Task List:**

1. **Retrieve information about the CTCF motif:**

   * Look up the CTCF binding motif in JASPAR
   * Identify:

     * Motif ID
     * Consensus sequence
     * Motif length
     * Matrix type (PWM/PFM)

2. **Download and inspect motif files:**

   * Download the motif in JASPAR format (PFM)
   * Load the file into Python
   * Convert the Position Frequency Matrix (PFM) into:

     * Position Weight Matrix (PWM)
     * Position-Specific Scoring Matrix (PSSM)
   * Inspect values and verify normalization

3. **Scan a DNA sequence for motif occurrences:**

   * Use the gene sequence obtained in Exercise 1
   * Slide the motif across the sequence
   * Compute a motif score at each position using the PSSM
   * Store positions and scores

4. **Plot motif score distribution:**

   * Create a histogram of motif scores
   * Plot score vs. sequence position
   * Identify patterns such as peaks or clusters

5. **Select significant motif hits:**

   * Define a threshold (e.g., top 1–5% scores or based on background distribution)
   * Filter motif matches above threshold
   * Record:

     * Start and end positions
     * Strand (+/- if scanning both)
     * Motif score
     * Matched sequence

6. **Export motif locations to BED format:**

   * Create a `.bed` file with the following columns:

     * Chromosome (or sequence ID)
     * Start position
     * End position
     * Motif name (e.g., CTCF)
     * Score
     * Strand
     * Matched sequence (optional)


---

### **Key Concepts:**

* Transcription factor binding motifs
* Position Frequency Matrix (PFM), PWM, and PSSM
* Motif scanning algorithms
* Score distributions and thresholding
* Genomic interval formats (BED)
* Biological interpretation of motif hits


In [ ]:
def load_jaspar_pfm(filepath):
    """
    Load a motif in JASPAR PFM format.

    Parameters:
        filepath (str): Path to the motif file

    Returns:
        dict: dictionary with keys 'A', 'C', 'G', 'T' and lists of frequencies
    """
    pfm = {}

    with open(filepath, "r") as f:
        lines = f.readlines()

    # HINT:
    # - Skip header lines (starting with '>')
    # - Each remaining line corresponds to a nucleotide (A, C, G, T)
    # - Extract numbers and store them as lists of floats

    # TODO: Parse file and fill pfm dictionary

    return pfm


# Example usage:
# pfm = load_jaspar_pfm("CTCF.jaspar")
# print(pfm)

In [ ]:
def pfm_to_pwm(pfm, pseudocount=1e-6):
    """
    Convert Position Frequency Matrix to Position Weight Matrix.

    HINT:
    - Normalize each column so that probabilities sum to 1
    - Add pseudocounts to avoid division by zero
    """
    pwm = None

    # TODO:
    # 1. Convert counts to probabilities per column
    # 2. Store in pwm dictionary

    return pwm


def pwm_to_pssm(pwm, background=0.25):
    """
    Convert PWM to PSSM using log2 odds.

    HINT:
    - PSSM = log2(pwm / background)
    - Background is typically 0.25 for each nucleotide
    """
    pssm = None

    # TODO:
    # 1. Compute log2 odds for each value
    # 2. Store in pssm dictionary

    return pssm

In [ ]:
def scan_sequence(sequence, pssm):
    """
    Scan a DNA sequence with a PSSM.

    Parameters:
        sequence (str): DNA sequence
        pssm (dict): Position-specific scoring matrix

    Returns:
        list of tuples: (position, score, subsequence)
    """
    results = []
    motif_length = len(pssm['A'])

    # HINT:
    # - Slide a window of motif_length across the sequence
    # - For each position:
    #     - Extract subsequence
    #     - Sum PSSM scores for each base
    # - Ignore invalid bases (e.g., 'N')

    # TODO: Implement sliding window scoring

    return results


# Example:
# hits = scan_sequence(sequence, pssm)

In [ ]:
def plot_score_distribution(scores):
    """
    Plot histogram of motif scores.
    """
    plt.figure(figsize=(8, 4))

    # TODO:
    # - Plot histogram of scores
    # - Add labels and title

    plt.show()


def plot_scores_along_sequence(results):
    """
    Plot motif scores along sequence positions.
    """
    positions = [r[0] for r in results]
    scores = [r[1] for r in results]

    plt.figure(figsize=(10, 4))

    # TODO:
    # - Scatter or line plot of positions vs scores
    # - Add labels and title

    plt.show()

In [ ]:
def select_significant_hits(results, percentile=95):
    """
    Select motif hits above a percentile threshold.

    HINT:
    - Extract all scores
    - Compute threshold using np.percentile
    - Filter results above threshold
    """
    scores = [r[1] for r in results]

    # TODO:
    # threshold = ...
    # significant = ...

    return significant, threshold


# Example:
# significant_hits, threshold = select_significant_hits(results)

In [ ]:
def save_to_bed(hits, output_file):
    """
    Save motif hits to BED format.

    BED format (6 columns):
    chrom, start, end, name, score, strand

    HINT:
    - start is 0-based
    - end = start + motif_length
    """
    with open(output_file, "w") as f:

        for pos, score, seq in hits:
            start = pos
            end = pos + len(seq)

            # TODO:
            # - Format line as tab-separated string
            # - Include motif name (e.g., "CTCF")
            # - Use '+' strand for now

            # f.write(...)

    print(f"Saved {len(hits)} hits to {output_file}")